# BirdCLEF 2026 — Pantanal Wildlife Audio Classification
### Inference Notebook

**How this works:**
1. Loads trained model weights from an uploaded Kaggle dataset
2. Slides a 5-second window over every test soundscape
3. Predicts probability of each of 234 species per window
4. Writes `submission.csv` to `/kaggle/working/`

**Expected Kaggle dataset structure:**
```
/kaggle/input/birdclef-2026/          ← competition data (auto-mounted)
/kaggle/input/birdclf2026-weights/    ← your uploaded checkpoints
    best_fold0.pt
    best_fold1.pt   (if available)
    ...
```

In [ ]:
# Install any packages not pre-installed on Kaggle
# librosa and timm are usually available — uncomment if needed
# !pip install -q librosa timm

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
import timm
from tqdm.notebook import tqdm

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {"cuda" if torch.cuda.is_available() else "cpu"}')

## Configuration
All paths and hyperparameters in one place — edit here if needed.

In [ ]:
import os

# ── Show full folder tree so we always know exact paths ─────────────
INPUT_DIR  = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')

print('Full /kaggle/input structure:')
for root, dirs, files in os.walk(INPUT_DIR):
    depth = str(root).count(os.sep) - str(INPUT_DIR).count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    if depth >= 2:   # stop drilling into audio files
        dirs.clear()

# ── Find competition folder — search all subdirs for taxonomy.csv ───
taxonomy_hits = list(INPUT_DIR.rglob('taxonomy.csv'))
assert len(taxonomy_hits) > 0, (
    'taxonomy.csv not found anywhere under /kaggle/input. '
    'Add BirdCLEF+ 2026 competition data via + Add Input.'
)
COMP_DIR = taxonomy_hits[0].parent
print(f'\nCompetition data : {COMP_DIR}')

# ── Find weights folder — search for any best_fold*.pt ──────────────
weight_hits = list(INPUT_DIR.rglob('best_fold*.pt'))
assert len(weight_hits) > 0, (
    'No best_fold*.pt found under /kaggle/input. '
    'Add birdclf2026-weights dataset via + Add Input.'
)
WEIGHTS_DIR = weight_hits[0].parent
print(f'Weights          : {WEIGHTS_DIR}')

TAXONOMY_CSV   = COMP_DIR / 'taxonomy.csv'
TEST_AUDIO_DIR = COMP_DIR / 'test_soundscapes'
SAMPLE_SUB_CSV = COMP_DIR / 'sample_submission.csv'

# ── Audio constants ─────────────────────────────────────────────────
SAMPLE_RATE = 32_000
WINDOW_SECS = 5
WINDOW_LEN  = SAMPLE_RATE * WINDOW_SECS
N_FFT       = 1024
HOP_LENGTH  = 320
N_MELS      = 128
FMIN        = 50
FMAX        = 14_000

# ── Inference settings ──────────────────────────────────────────────
BATCH_SIZE = 64
NUM_WORKERS = 2
USE_TTA    = True
N_TTA      = 3
MODEL_NAME = 'efficientnet_b3'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {DEVICE}')

## Species List
Load from taxonomy.csv — this defines the column order in submission.csv.

In [ ]:
taxonomy_df  = pd.read_csv(TAXONOMY_CSV)
SPECIES_LIST = taxonomy_df['primary_label'].astype(str).tolist()
NUM_CLASSES  = len(SPECIES_LIST)
print(f'Species: {NUM_CLASSES}')
print(f'First 5: {SPECIES_LIST[:5]}')

## Audio & Spectrogram Utilities

In [ ]:
def audio_to_melspec(audio: np.ndarray) -> np.ndarray:
    """Convert waveform to normalized log-mel spectrogram."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS,
        fmin=FMIN, fmax=FMAX,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel.astype(np.float32)


def spec_augment(spec: np.ndarray,
                 num_freq: int = 2, freq_size: int = 15,
                 num_time: int = 2, time_size: int = 30) -> np.ndarray:
    """SpecAugment — used during TTA."""
    spec = spec.copy()
    n_mels, n_time = spec.shape
    for _ in range(num_freq):
        f0 = random.randint(0, n_mels - freq_size)
        spec[f0:f0 + freq_size, :] = 0.0
    for _ in range(num_time):
        t0 = random.randint(0, n_time - time_size)
        spec[:, t0:t0 + time_size] = 0.0
    return spec

## Soundscape Dataset
Slices each test recording into non-overlapping 5-second windows.
Row ID format: `{filename_stem}_{end_second}`

In [ ]:
class SoundscapeDataset(Dataset):
    """
    Loads one soundscape file and returns all 5-second windows.
    row_id matches the submission format exactly.
    """
    def __init__(self, filepath: Path):
        self.stem = filepath.stem
        audio, _ = librosa.load(str(filepath), sr=SAMPLE_RATE, mono=True)

        self.windows = []
        for start in range(0, len(audio), WINDOW_LEN):
            chunk = audio[start:start + WINDOW_LEN]
            if len(chunk) < WINDOW_LEN:
                chunk = np.pad(chunk, (0, WINDOW_LEN - len(chunk)))
            end_sec = (start // WINDOW_LEN + 1) * WINDOW_SECS
            self.windows.append((chunk, end_sec))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        audio, end_sec = self.windows[idx]
        spec = audio_to_melspec(audio)
        return {
            'spectrogram': torch.from_numpy(spec).unsqueeze(0),
            'row_id': f'{self.stem}_{end_sec}',
        }

## Model Definition

In [ ]:
class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes: int, model_name: str = 'efficientnet_b3',
                 pretrained: bool = False, drop_rate: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,   # False — weights come from our checkpoint
            in_chans=1,
            num_classes=0,
            global_pool='avg',
            drop_rate=drop_rate,
        )
        feature_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


class CNN14Block(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
        )
        self.pool = nn.AvgPool2d(2)

    def forward(self, x):
        return self.pool(self.conv(x))


class CNN14Classifier(nn.Module):
    def __init__(self, num_classes: int, drop_rate: float = 0.3):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(1)
        self.layers = nn.Sequential(
            CNN14Block(1, 64),   CNN14Block(64, 128),
            CNN14Block(128, 256), CNN14Block(256, 512),
            CNN14Block(512, 1024), CNN14Block(1024, 2048),
        )
        self.dropout = nn.Dropout(drop_rate)
        self.head = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.bn0(x)
        x = self.layers(x).mean(dim=[2, 3])
        return self.head(self.dropout(x))


def load_model(checkpoint_path: Path, model_name: str, num_classes: int, device):
    """Load a trained checkpoint."""
    if model_name == 'efficientnet_b3':
        model = EfficientNetClassifier(num_classes, pretrained=False)
    else:
        model = CNN14Classifier(num_classes)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    return model.to(device)


print('Model classes defined.')

## Load Checkpoints
Loads all available fold checkpoints from your uploaded dataset and ensembles them.

In [ ]:
checkpoints = sorted(WEIGHTS_DIR.glob('best_fold*.pt'))
print(f'Found {len(checkpoints)} checkpoint(s):')
for c in checkpoints:
    print(f'  {c.name}')

assert len(checkpoints) > 0, (
    f'No checkpoints found in {WEIGHTS_DIR}. '
    f'Upload your best_fold*.pt files as a Kaggle dataset named birdclf2026-weights.'
)

models = []
for ckpt in checkpoints:
    m = load_model(ckpt, MODEL_NAME, NUM_CLASSES, DEVICE)
    models.append(m)
    print(f'  Loaded {ckpt.name}')

print(f'\nEnsemble ready: {len(models)} model(s)')

## Inference
Slides a 5-second window over every test soundscape and averages predictions across all models + TTA.

In [ ]:
@torch.no_grad()
def predict_batch(models, specs, device, use_tta=True, n_tta=3):
    """
    Run all models on a batch, average their sigmoid outputs.
    Optionally average over TTA augmentations too.
    """
    specs = specs.to(device)
    all_probs = []

    for model in models:
        # Original prediction
        probs = torch.sigmoid(model(specs)).cpu().numpy()
        all_probs.append(probs)

        # TTA — run with SpecAugment and average
        if use_tta:
            for _ in range(n_tta):
                aug = torch.stack([
                    torch.from_numpy(
                        spec_augment(s.squeeze(0).numpy())
                    ).unsqueeze(0)
                    for s in specs.cpu()
                ]).to(device)
                all_probs.append(torch.sigmoid(model(aug)).cpu().numpy())

    # Average across all models × TTA versions
    return np.mean(all_probs, axis=0)   # (batch, num_classes)


def run_inference(models, test_dir, species_list, device,
                  batch_size=64, use_tta=True, n_tta=3):
    """Process all test soundscapes, return full predictions DataFrame."""
    audio_exts = {'.ogg', '.flac', '.wav', '.mp3'}
    files = sorted([f for f in test_dir.rglob('*') if f.suffix in audio_exts])
    print(f'Test soundscapes found: {len(files)}')

    all_row_ids, all_probs = [], []

    for filepath in tqdm(files, desc='Soundscapes'):
        ds = SoundscapeDataset(filepath)
        loader = DataLoader(ds, batch_size=batch_size,
                            shuffle=False, num_workers=NUM_WORKERS)

        for batch in loader:
            probs = predict_batch(models, batch['spectrogram'],
                                  device, use_tta, n_tta)
            all_row_ids.extend(batch['row_id'])
            all_probs.append(probs)

    probs_matrix = np.concatenate(all_probs)         # (N_windows, 234)
    df = pd.DataFrame(probs_matrix, columns=species_list)
    df.insert(0, 'row_id', all_row_ids)
    return df


print('Inference functions ready.')

In [ ]:
# Run inference — this is the slow cell, ~minutes depending on number of soundscapes
predictions_df = run_inference(
    models,
    test_dir=TEST_AUDIO_DIR,
    species_list=SPECIES_LIST,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    use_tta=USE_TTA,
    n_tta=N_TTA,
)

print(f'Predictions shape: {predictions_df.shape}')
predictions_df.head(3)

## Write Submission
Align columns exactly to sample_submission.csv and save.

In [ ]:
# Load sample submission to get exact column order
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
expected_columns = sample_sub.columns.tolist()   # ['row_id', 'species1', ...]

# Reorder to match exactly
submission = predictions_df[expected_columns]

# Save
output_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {submission.shape}  ({submission.shape[0]} windows × {submission.shape[1]} columns)')
submission.head(3)

## Sanity Checks

In [ ]:
# 1. No missing values
assert submission.isnull().sum().sum() == 0, 'Missing values found!'

# 2. All probabilities between 0 and 1
prob_cols = [c for c in submission.columns if c != 'row_id']
assert submission[prob_cols].min().min() >= 0.0, 'Negative probabilities found!'
assert submission[prob_cols].max().max() <= 1.0, 'Probabilities > 1 found!'

# 3. Columns match sample submission exactly
assert list(submission.columns) == expected_columns, 'Column mismatch!'

print('All checks passed ✓')
print(f'  Rows    : {len(submission)}')
print(f'  Columns : {len(submission.columns)}')
print(f'  Prob range: [{submission[prob_cols].min().min():.4f}, {submission[prob_cols].max().max():.4f}]')